# MultiLogiEval Two-Stage: Error Classification Analysis

Comprehensive analysis of 41 verified-but-wrong cases using error root cause classification.

Compares Two-Stage error patterns with simple Lean approach.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Define consistent color palette
ERROR_COLORS = {
    'AXIOMATIZES_CONCLUSION': '#e74c3c',
    'AXIOMATIZES_CONTRADICTION': '#3498db',
    'INCORRECT_FORMALIZATION': '#f39c12',
    'REASONING_FAILURE': '#2ecc71',
    'AXIOMATIZES_UNMENTIONED': '#9b59b6',
    'OTHER': '#95a5a6'
}

## 1. Load Data and Basic Statistics

In [ ]:
# Load both Two-Stage and simple Lean data for comparison
df_ts = pd.read_csv('../results/multilogieval/all_depths/two_stage/error_root_cause_analysis.csv')
df_lean = pd.read_csv('../results/multilogieval/all_depths/lean/error_root_cause_analysis.csv')

print(f"Two-Stage cases: {len(df_ts)}")
print(f"Simple Lean cases: {len(df_lean)}")

if len(df_ts) < len(df_lean):
    print(f"Improvement: {len(df_lean) - len(df_ts)} fewer errors ({(len(df_lean)-len(df_ts))/len(df_lean)*100:.1f}% reduction)")
elif len(df_ts) > len(df_lean):
    print(f"Regression: {len(df_ts) - len(df_lean)} more errors ({(len(df_ts)-len(df_lean))/len(df_lean)*100:.1f}% increase)")
else:
    print(f"Same number of errors")

print(f"\nColumns: {list(df_ts.columns)}")
print(f"\nLogic types: {sorted(df_ts['Logic_Type'].unique())}")
print(f"Depth levels: {sorted(df_ts['Depth'].unique())}")

In [ ]:
# Show first few rows
df_ts.head()

## 2. Error Type Distribution

In [ ]:
# Calculate error distribution for both approaches
ts_dist = df_ts['Root_Cause_Category'].value_counts()
lean_dist = df_lean['Root_Cause_Category'].value_counts()

print("Two-Stage vs Simple Lean:")
print("=" * 70)

all_categories = sorted(set(ts_dist.index) | set(lean_dist.index), 
                       key=lambda x: lean_dist.get(x, 0) + ts_dist.get(x, 0), reverse=True)

for category in all_categories:
    ts_count = ts_dist.get(category, 0)
    lean_count = lean_dist.get(category, 0)
    ts_pct = (ts_count / len(df_ts)) * 100 if len(df_ts) > 0 else 0
    lean_pct = (lean_count / len(df_lean)) * 100 if len(df_lean) > 0 else 0
    diff = lean_count - ts_count
    
    print(f"{category:30s}")
    print(f"  Simple Lean: {lean_count:3d} cases ({lean_pct:5.1f}%)")
    print(f"  Two-Stage:   {ts_count:3d} cases ({ts_pct:5.1f}%)")
    if diff > 0:
        print(f"  ✓ Two-Stage improved: {diff} fewer cases")
    elif diff < 0:
        print(f"  ✗ Two-Stage worse: {abs(diff)} more cases")
    else:
        print(f"  ~ Same performance")
    print()

# Calculate axiomatization totals
axiom_types = ['AXIOMATIZES_CONCLUSION', 'AXIOMATIZES_CONTRADICTION', 'AXIOMATIZES_UNMENTIONED']
ts_axiom = sum(ts_dist.get(t, 0) for t in axiom_types)
lean_axiom = sum(lean_dist.get(t, 0) for t in axiom_types)

print("=" * 70)
print(f"TOTAL AXIOMATIZATION ERRORS:")
print(f"  Simple Lean: {lean_axiom} cases ({lean_axiom/len(df_lean)*100:.1f}%)")
print(f"  Two-Stage:   {ts_axiom} cases ({ts_axiom/len(df_ts)*100:.1f}%)")
if lean_axiom > ts_axiom:
    print(f"  Improvement: {lean_axiom - ts_axiom} cases ({(lean_axiom - ts_axiom)/lean_axiom*100:.1f}% reduction)")
elif lean_axiom < ts_axiom:
    print(f"  Regression: {ts_axiom - lean_axiom} more cases")
else:
    print(f"  Same performance")

In [ ]:
# Visualize error distribution comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart - Two-Stage
colors = [ERROR_COLORS.get(cat, '#95a5a6') for cat in ts_dist.index]
ts_dist.plot(kind='bar', ax=ax1, color=colors, edgecolor='black', linewidth=1.2)
ax1.set_title('Two-Stage Error Distribution', fontsize=14, fontweight='bold')
ax1.set_xlabel('Error Category', fontsize=12)
ax1.set_ylabel('Number of Cases', fontsize=12)
ax1.tick_params(axis='x', rotation=45)
ax1.grid(axis='y', alpha=0.3)

# Add count labels
for i, v in enumerate(ts_dist.values):
    ax1.text(i, v + 0.3, str(v), ha='center', va='bottom', fontweight='bold')

# Pie chart
colors_pie = [ERROR_COLORS.get(cat, '#95a5a6') for cat in ts_dist.index]
ax2.pie(ts_dist.values, labels=ts_dist.index, autopct='%1.1f%%', 
        colors=colors_pie, startangle=90, textprops={'fontsize': 10})
ax2.set_title('Two-Stage Error Distribution (Pie Chart)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side comparison
fig, ax = plt.subplots(figsize=(14, 6))

# Create comparison dataframe
comparison_df = pd.DataFrame({
    'Simple Lean': lean_dist,
    'Two-Stage': ts_dist
}).fillna(0)

comparison_df.plot(kind='bar', ax=ax, edgecolor='black', linewidth=1.2)
ax.set_title('Error Type Comparison: Simple Lean vs Two-Stage', fontsize=14, fontweight='bold')
ax.set_xlabel('Error Category', fontsize=12)
ax.set_ylabel('Number of Cases', fontsize=12)
ax.legend(title='Approach', fontsize=11)
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Error Types by Prediction Pattern

In [ ]:
# Cross-tabulation for Two-Stage
pattern_error = pd.crosstab(df_ts['Pattern'], df_ts['Root_Cause_Category'])

print("Two-Stage Error Types by Pattern:")
print("=" * 70)
print(pattern_error)

print("\nKey insights:")
for pattern in pattern_error.index:
    top_error = pattern_error.loc[pattern].idxmax()
    count = pattern_error.loc[pattern].max()
    total = pattern_error.loc[pattern].sum()
    print(f"  {pattern} ({total} total): Most common is {top_error} ({count} cases)")

In [ ]:
# Visualize grouped bar chart
fig, ax = plt.subplots(figsize=(14, 6))

# Reorder columns by error type frequency
ordered_cols = ts_dist.index.tolist()
pattern_error_ordered = pattern_error[ordered_cols]

pattern_error_ordered.plot(kind='bar', ax=ax, 
                           color=[ERROR_COLORS.get(col, '#95a5a6') for col in ordered_cols],
                           edgecolor='black', linewidth=1.2)
ax.set_title('Two-Stage: Error Types by Prediction Pattern', fontsize=14, fontweight='bold')
ax.set_xlabel('Prediction Pattern (Ground Truth → Prediction)', fontsize=12)
ax.set_ylabel('Number of Cases', fontsize=12)
ax.legend(title='Error Type', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Error Distribution by Logic Type and Depth

In [ ]:
# Error distribution by logic type
logic_error = pd.crosstab(df_ts['Logic_Type'], df_ts['Root_Cause_Category'])

print("Error Types by Logic Type:")
print("=" * 70)
print(logic_error)

# Error distribution by depth
depth_error = pd.crosstab(df_ts['Depth'], df_ts['Root_Cause_Category'])

print("\nError Types by Depth:")
print("=" * 70)
print(depth_error)

In [ ]:
# Visualize logic type and depth distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# By logic type
ordered_cols = ts_dist.index.tolist()
logic_error_ordered = logic_error[ordered_cols]
logic_error_ordered.plot(kind='bar', ax=ax1,
                         color=[ERROR_COLORS.get(col, '#95a5a6') for col in ordered_cols],
                         edgecolor='black', linewidth=1.2)
ax1.set_title('Two-Stage: Error Types by Logic Type', fontsize=14, fontweight='bold')
ax1.set_xlabel('Logic Type', fontsize=12)
ax1.set_ylabel('Number of Cases', fontsize=12)
ax1.legend(title='Error Type', bbox_to_anchor=(1.05, 1), loc='upper left')
ax1.tick_params(axis='x', rotation=0)
ax1.grid(axis='y', alpha=0.3)

# By depth
depth_error_ordered = depth_error[ordered_cols]
depth_error_ordered.plot(kind='bar', ax=ax2,
                         color=[ERROR_COLORS.get(col, '#95a5a6') for col in ordered_cols],
                         edgecolor='black', linewidth=1.2)
ax2.set_title('Two-Stage: Error Types by Depth', fontsize=14, fontweight='bold')
ax2.set_xlabel('Depth Level', fontsize=12)
ax2.set_ylabel('Number of Cases', fontsize=12)
ax2.legend(title='Error Type', bbox_to_anchor=(1.05, 1), loc='upper left')
ax2.tick_params(axis='x', rotation=0)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Key Findings

In [ ]:
print("=" * 70)
print("KEY FINDINGS: MULTILOGIEVAL TWO-STAGE vs SIMPLE LEAN")
print("=" * 70)

print(f"\nOVERALL PERFORMANCE:")
print(f"  Simple Lean: {len(df_lean)} verified-but-wrong cases")
print(f"  Two-Stage:   {len(df_ts)} verified-but-wrong cases")
if len(df_ts) < len(df_lean):
    print(f"  ✓ Two-Stage reduces errors by {len(df_lean) - len(df_ts)} cases ({(len(df_lean)-len(df_ts))/len(df_lean)*100:.1f}%)")
elif len(df_ts) > len(df_lean):
    print(f"  ✗ Two-Stage increases errors by {len(df_ts) - len(df_lean)} cases ({(len(df_ts)-len(df_lean))/len(df_lean)*100:.1f}%)")

print(f"\n1. AXIOMATIZATION ERRORS:")
print(f"   Simple Lean: {lean_axiom} cases ({lean_axiom/len(df_lean)*100:.1f}%)")
print(f"   Two-Stage:   {ts_axiom} cases ({ts_axiom/len(df_ts)*100:.1f}%)")
if lean_axiom > ts_axiom:
    reduction_pct = (lean_axiom - ts_axiom) / lean_axiom * 100
    print(f"   ✓ Two-Stage reduces axiomatization by {reduction_pct:.1f}%")
elif lean_axiom < ts_axiom:
    print(f"   ✗ Two-Stage increases axiomatization errors")

# Compare individual error types
error_comparisons = [
    ('AXIOMATIZES_CONCLUSION', '2. AXIOMATIZES_CONCLUSION'),
    ('INCORRECT_FORMALIZATION', '3. INCORRECT_FORMALIZATION'),
    ('REASONING_FAILURE', '4. REASONING_FAILURE'),
    ('AXIOMATIZES_CONTRADICTION', '5. AXIOMATIZES_CONTRADICTION')
]

for error_type, label in error_comparisons:
    lean_count = lean_dist.get(error_type, 0)
    ts_count = ts_dist.get(error_type, 0)
    
    print(f"\n{label}:")
    print(f"   Simple Lean: {lean_count} cases ({lean_count/len(df_lean)*100:.1f}%)")
    print(f"   Two-Stage:   {ts_count} cases ({ts_count/len(df_ts)*100:.1f}%)")
    
    if lean_count > ts_count:
        print(f"   ✓ Improvement ({lean_count - ts_count} fewer cases)")
    elif lean_count < ts_count:
        if lean_count > 0:
            print(f"   ✗ Two-Stage has {ts_count/lean_count:.1f}x MORE errors")
        else:
            print(f"   ✗ Two-Stage introduces {ts_count} new errors")
    else:
        print(f"   ~ Similar performance")

print(f"\n6. LOGIC TYPE BREAKDOWN:")
for logic in sorted(df_ts['Logic_Type'].unique()):
    ts_logic_count = len(df_ts[df_ts['Logic_Type'] == logic])
    lean_logic_count = len(df_lean[df_lean['Logic_Type'] == logic])
    print(f"   {logic}: Lean={lean_logic_count}, Two-Stage={ts_logic_count}")

print("\n" + "=" * 70)
print("CONCLUSION:")
print("=" * 70)
if len(df_ts) < len(df_lean):
    print(f"\nTwo-Stage shows overall improvement with {len(df_lean) - len(df_ts)} fewer errors.")
else:
    print(f"\nTwo-Stage shows mixed results with {len(df_ts) - len(df_lean)} more errors overall.")

print(f"\nAcross {df_ts['Logic_Type'].nunique()} logic types and {df_ts['Depth'].nunique()} depth levels,")
print("Two-Stage reasoning affects error distribution patterns.")
print("=" * 70)

## 6. Example Cases from Top 3 Error Types

In [ ]:
# Get top 3 error types for Two-Stage
top_3_types = ts_dist.head(3).index.tolist()

for error_type in top_3_types:
    subset = df_ts[df_ts['Root_Cause_Category'] == error_type]
    count = len(subset)
    pct = count / len(df_ts) * 100
    
    print("\n" + "=" * 70)
    print(f"{error_type} ({count} cases, {pct:.1f}%)")
    print("=" * 70)
    
    # Show up to 3 examples
    for idx, (_, row) in enumerate(subset.head(3).iterrows(), 1):
        print(f"\nExample {idx}:")
        print(f"  Logic: {row['Logic_Type']}, Depth: {row['Depth']}")
        print(f"  Pattern: {row['Pattern']}")
        print(f"  Context: {str(row['Context'])[:120]}...")
        print(f"  Question: {str(row['Question'])[:120]}...")
        if pd.notna(row['Problematic_Lines']):
            print(f"  Problematic Line: {row['Problematic_Lines']}")
        print(f"  Error: {str(row['Error_Description'])[:150]}...")